# 🧠 Building Micrograd from Scratch
### A step-by-step implementation of an autograd engine and backpropagation

In this notebook, I build a mini deep learning engine from scratch — inspired by how frameworks like PyTorch work internally.

Instead of using high-level libraries, I:
- Build an **automatic differentiation (autograd) engine**
- Implement **backpropagation manually**
- Create a small **neural network (MLP)**
- Train it using **gradient descent**

> 📌 Based on Andrej Karpathy's [micrograd lecture](https://www.youtube.com/watch?v=VMj-3S1tku0)

## 🎯 Why Build This From Scratch?

Modern frameworks like PyTorch hide the complexity of training neural networks behind clean APIs.

But under the hood:
- Neural networks are just **mathematical expressions**
- Backpropagation applies the **chain rule** from calculus
- Gradients flow backward through a **computational graph**

Building everything from scratch makes all of this concrete and visible.

## 🔹 What is Micrograd?

Micrograd is a tiny autograd engine that implements backpropagation over a dynamically built computational graph.

It works by:
- Representing every computation as a **node in a graph**
- **Tracking which operation** produced each value
- Computing gradients using **reverse-mode automatic differentiation**

Even though it operates on scalar values only, it is powerful enough to build and train full neural networks.

In [ ]:
# Importing necessary libraries
import numpy as np
import math
import matplotlib.pyplot as plt
import random

## 🔹 Step 1: Building the Value Object

A neural network is just a mathematical expression — it takes data as input, performs operations, and produces an output.

Micrograd is a **scalar-level autograd engine**. It tracks every operation so that gradients can be computed automatically during backpropagation.

We start by building a `Value` class that:
- Stores a **number** (`data`)
- Stores its **gradient** (`grad`)
- Tracks **which operation** created it (`_op`)
- Remembers its **parent nodes** (`_prev`) for backpropagation

Each operation like `+` or `*` creates a new `Value` and links it back to its inputs — automatically building the computational graph.

In [ ]:
class Value:
    def __init__(self, data):
        self.data = data
    
    # Controls how Value prints — makes debugging easier
    def __repr__(self):
        return f"Value(data={self.data})"

# Quick check
a = Value(2.0)
print(a)

## ➕ Adding Arithmetic Operations

Now we add `+` and `*` to the `Value` class so we can build expressions.

We also start tracking:
- `_prev` — which Values produced this one (the children)
- `_op` — which operation was used

This is what allows us to reconstruct the full computational graph later.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        return out

a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a * b; e.label = 'e'
d = e + c; d.label = 'd'
print(d)
print("Children of d:", d._prev)
print("Operation that produced d:", d._op)

## 🔍 Visualizing the Computational Graph

Before we do backpropagation, it helps to *see* the graph we're building.

We use **Graphviz** to draw each Value as a node, and each operation as an edge connecting inputs to outputs.

> Note: The `trace` and `draw_dot` functions below are utility code — their purpose is visualization only, not part of the core autograd engine.

In [ ]:
from graphviz import Digraph

def trace(root):
    # Builds the complete set of nodes and edges by traversing the graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})  # Left to right layout
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

# Building a slightly deeper expression to visualize
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a * b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label='f')
L = d * f; L.label = 'L'

draw_dot(L)

## ↩️ Step 2: Backpropagation

The diagram above shows the **forward pass** — how values flow left to right to produce the output `L`.

Now we run **backpropagation** — starting from `L` and working backwards to compute how much each input contributed to the output.

For every node in the graph, we want to compute:

$$\frac{\partial L}{\partial \text{node}}$$

This tells us how sensitive the output `L` is to each value — which is exactly what we need to train a neural network.

## 📐 Manually Computing Gradients

Before automating this, let's verify gradients by **numerical estimation** — nudging each variable by a tiny amount `h` and seeing how much `L` changes.

This is the definition of a derivative:

$$\frac{df}{dx} \approx \frac{f(x+h) - f(x)}{h}$$

In [ ]:
def estimate_gradients_numerically():
    h = 0.001

    # Base computation
    a = Value(2.0); b = Value(-3.0); c = Value(10.0)
    e = a * b; d = e + c; f = Value(-2.0); L = d * f
    L1 = L.data

    # Gradient of a
    a = Value(2.0); a.data += h
    b = Value(-3.0); c = Value(10.0)
    e = a * b; d = e + c; f = Value(-2.0); L = d * f
    print("a grad:", (L.data - L1) / h)

    # Gradient of b
    a = Value(2.0); b = Value(-3.0); b.data += h
    c = Value(10.0)
    e = a * b; d = e + c; f = Value(-2.0); L = d * f
    print("b grad:", (L.data - L1) / h)

    # Gradient of f
    a = Value(2.0); b = Value(-3.0); c = Value(10.0)
    e = a * b; d = e + c; f = Value(-2.0); f.data += h; L = d * f
    print("f grad:", (L.data - L1) / h)

estimate_gradients_numerically()

## 🧬 Step 3: Manual Backpropagation Through a Neuron

A neural network is made of **layers of neurons**. A single neuron:

1. Takes inputs `x1, x2, ...`
2. Multiplies each by a weight `w1, w2, ...`
3. Adds a bias `b`
4. Passes the result through an **activation function**

The activation function "squashes" the output to a fixed range. We use **tanh** here, which squashes values to `[-1, 1]`.

In [ ]:
# Visualizing tanh activation function
x = np.arange(-5, 5, 0.2)
plt.plot(x, np.tanh(x))
plt.title("tanh activation function")
plt.xlabel("x")
plt.ylabel("tanh(x)")
plt.grid()
plt.show()

## ⚙️ Automating Backpropagation with `_backward`

Instead of computing gradients manually, we define a `_backward` function inside each operation.

Each `_backward`:
- Knows the **local derivative** of its operation
- Multiplies it by the **incoming gradient** (chain rule)
- Accumulates the result into the input's `.grad`

We use `+=` instead of `=` to handle cases where a variable is used more than once in an expression — otherwise we'd overwrite its gradient instead of accumulating it.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "Only int/float powers supported"
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def __radd__(self, other):  # ← add this too, handles 2+a
        return self + other

    def __truediv__(self, other):
        return self * other ** -1

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [ ]:
# Single neuron: x1w1 + x2w2 + b passed through tanh
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')

x1w1 = x1 * w1; x1w1.label = 'x1w1'
x2w2 = x2 * w2; x2w2.label = 'x2w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1 + x2w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

o.backward()
draw_dot(o)

In [ ]:
# tanh can also be expressed using primitive operations (exp, sub, div)
# This shows that micrograd doesn't need tanh built-in — it emerges from simpler ops
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')

x1w1 = x1 * w1; x1w1.label = 'x1w1'
x2w2 = x2 * w2; x2w2.label = 'x2w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1 + x2w2'
n = x1w1x2w2 + b; n.label = 'n'

e = (2 * n).exp(); e.label = 'e'
o = (e - 1) / (e + 1); o.label = 'o'

o.backward()
draw_dot(o)

## 🔥 Verification: Comparing with PyTorch

Let's verify our gradients are correct by running the same expression in PyTorch and comparing results.

PyTorch uses `float64` (double precision) to match Python's default float precision — we need to explicitly cast to avoid rounding differences.

In [ ]:
import torch

# PyTorch uses float64 (double) to match Python's default precision
x1 = torch.Tensor([2.0]).double();  x1.requires_grad = True
x2 = torch.Tensor([0.0]).double();  x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double(); w1.requires_grad = True
w2 = torch.Tensor([1.0]).double();  w2.requires_grad = True
b  = torch.Tensor([6.8813735870195432]).double(); b.requires_grad = True

n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

o.backward()

print("PyTorch gradients:")
print(f"  x1: {x1.grad.item():.4f}")
print(f"  w1: {w1.grad.item():.4f}")
print(f"  x2: {x2.grad.item():.4f}")
print(f"  w2: {w2.grad.item():.4f}")

## 🏗️ Step 4: Building a Neural Network

Now that we have a working autograd engine, we can use it to build actual neural networks.

We build three classes on top of `Value`:

- **`Neuron`** — a single unit with weights, bias, and tanh activation
- **`Layer`** — a collection of neurons that all receive the same inputs
- **`MLP`** (Multi-Layer Perceptron) — a stack of layers forming a complete network

This is exactly how PyTorch's `nn.Module` works — just at a much higher level.

In [ ]:
class Neuron:
    def __init__(self, nin):
        # nin: number of inputs to this neuron
        # Weights initialized randomly between -1 and 1
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        # Computes: tanh(w · x + b)
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        # nout: number of neurons in this layer
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        # Return a single value if only one output neuron, otherwise return list
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        # nouts: list of layer sizes, e.g. [4, 4, 1]
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


# Quick test
x = [2.0, 3.0, -1.0]          # 3 inputs
n = MLP(3, [4, 4, 1])          # Two hidden layers of 4, one output
print("Output:", n(x))
print("Total parameters:", len(n.parameters()))

## 📊 Step 5: A Small Binary Classification Dataset

We define a tiny dataset with 4 examples and binary targets (`+1` or `-1`).

The goal: train the MLP to predict the correct target for each input.

In [ ]:
xs = [
    [2.0,  3.0, -1.0],
    [3.0, -1.0,  0.5],
    [0.5,  1.0,  1.0],
    [1.0,  1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]  # Desired targets

# Forward pass — see where we start
ypred = [n(x) for x in xs]
print("Predictions:", [round(y.data, 4) for y in ypred])
print("Targets:    ", ys)

## 📉 Step 6: Loss Function (Mean Squared Error)

To train the network, we need a single number that measures how wrong the predictions are. This is called the **loss**.

We use **Mean Squared Error (MSE)**:

$$\text{loss} = \sum_{i} (\hat{y}_i - y_i)^2$$

- When predictions are close to targets → loss is small
- When predictions are far from targets → loss is large

The goal of training is to **minimize this loss**.

In [ ]:
loss = sum(((yout - ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0.0))
print("Loss:", loss.data)

## 🔄 Step 7: Gradient Descent — The Training Loop

Now we put everything together into a training loop:

1. **Forward pass** — compute predictions and loss
2. **Zero gradients** — reset all `.grad` to `0` before each backward pass
3. **Backward pass** — compute gradients via `loss.backward()`
4. **Update** — nudge each parameter in the direction that reduces loss

The update rule is:

$$p \leftarrow p - \text{lr} \times \frac{\partial \text{loss}}{\partial p}$$

We subtract the gradient (not add) because we want to move *downhill* on the loss surface.

> ⚠️ Zeroing gradients before each backward pass is critical. Without it, gradients from previous steps accumulate and corrupt the update.

In [ ]:
# Reset the network with fresh random weights
n = MLP(3, [4, 4, 1])

xs = [
    [2.0,  3.0, -1.0],
    [3.0, -1.0,  0.5],
    [0.5,  1.0,  1.0],
    [1.0,  1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

for k in range(20):
    # Forward pass
    ypred = [n(x) for x in xs]
    loss = sum(((yout - ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0.0))

    # Zero gradients before backward pass
    for p in n.parameters():
        p.grad = 0.0

    # Backward pass
    loss.backward()

    # Update parameters
    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(f"Step {k:02d} | Loss: {loss.data:.4f}")

In [ ]:
print("\nFinal Predictions vs Targets:")
print(f"{'Input':<25} {'Predicted':>10} {'Target':>10}")
print("-" * 47)
for x, y in zip(xs, ys):
    pred = n(x).data
    print(f"{str(x):<25} {pred:>10.4f} {y:>10.1f}")

## ✅ Summary

In this notebook, I built a complete deep learning engine from scratch:

| Component | What it does |
|---|---|
| `Value` | Stores data + gradient, tracks computation graph |
| `_backward` | Computes local gradients using chain rule |
| `backward()` | Runs full backprop via topological sort |
| `Neuron` | Single unit: weighted sum + tanh |
| `Layer` | Collection of neurons |
| `MLP` | Stacked layers forming a full network |
| Training loop | Forward → zero grad → backward → update |

### Key Takeaways

- **Autograd** is just careful bookkeeping of operations and their derivatives
- **Backpropagation** is the chain rule applied recursively over a computation graph
- **Gradient descent** nudges parameters downhill on the loss surface
- PyTorch does all of this under the hood — now we know what's happening

> The full source for micrograd is available at [github.com/karpathy/micrograd](https://github.com/karpathy/micrograd)